<a href="https://colab.research.google.com/github/luzcielponce-create/MS-E-226-Project-/blob/main/Holdout_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#set up & load data
# Project 2 setup: load development and holdout data
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, log_loss, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from statsmodels.stats.multitest import multipletests

DATA_DIR = Path("/content")
DEV_PATH = DATA_DIR / "explore_80.csv"
HOLDOUT_PATH = DATA_DIR / "holdout_20.csv"
SEED = 226
BOOTSTRAP_REPS = 500
THRESHOLD = 0.5

for path in (DEV_PATH, HOLDOUT_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Cannot find {path}. Upload both CSVs into sample_data.")

dev = pd.read_csv(DEV_PATH).replace(r"^\s*$", np.nan, regex=True)
holdout = pd.read_csv(HOLDOUT_PATH).replace(r"^\s*$", np.nan, regex=True)
y_dev = dev["Netflix"].astype(int)
y_holdout = holdout["Netflix"].astype(int)
X_dev = dev.drop(columns="Netflix").copy()
X_holdout = holdout.drop(columns="Netflix").copy()

# Include every Part 1 genre indicator, including genre_Adult (28 total).
genre_columns = sorted(c for c in X_dev.columns if c.startswith("genre_"))
if len(genre_columns) != 28:
    raise ValueError(f"Expected 28 genre indicators; found {len(genre_columns)}: {genre_columns}")
missing = set(genre_columns) - set(X_holdout.columns)
if missing:
    raise ValueError(f"Holdout is missing genre indicators: {sorted(missing)}")

print(f"Development rows: {len(dev):,}")
print(f"Holdout rows: {len(holdout):,}")
print(f"Genre indicators: {len(genre_columns)}")


Development rows: 11,906
Holdout rows: 2,977
Genre indicators: 28


In [7]:
# Project 2, Section 2: refit selected Part 1 pipeline on development data,
# then evaluate it on the untouched holdout.

class PrepareTitleFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, genre_columns):
        self.genre_columns = genre_columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        out = X.copy().replace(r"^\s*$", np.nan, regex=True)
        out["Age"] = out["Age"].fillna("unrated")
        out["genres_missing"] = out["genres"].isna().astype(int)
        out.loc[out["genres_missing"] == 1, self.genre_columns] = 0

        rt_text = out["Rotten Tomatoes"].astype("string")
        rt_text = rt_text.str.replace("/100", "", regex=False)
        out["rt_score"] = pd.to_numeric(rt_text, errors="coerce")
        for col in ("imdb_rating", "imdb_num_votes"):
            out[col] = pd.to_numeric(out[col], errors="coerce")
        return out


class RuntimeMedianImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        runtime = pd.to_numeric(X["runtime_minutes"], errors="coerce")
        self.medians_ = runtime.groupby(X["content_type"]).median()
        self.fallback_ = runtime.median()
        if pd.isna(self.fallback_):
            raise ValueError("No observed training runtimes.")
        return self

    def transform(self, X):
        out = X.copy()
        runtime = pd.to_numeric(out["runtime_minutes"], errors="coerce")
        out["runtime_missing"] = runtime.isna().astype(int)
        replacements = out["content_type"].map(self.medians_).fillna(self.fallback_)
        out["runtime_minutes"] = runtime.fillna(replacements)
        return out


numeric_columns = ["runtime_minutes", "imdb_rating", "rt_score", "imdb_num_votes", "Year"]
binary_columns = ["Type", "runtime_missing", "genres_missing"] + genre_columns

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
])

encoding = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("age", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Age"]),
        ("binary", "passthrough", binary_columns),
    ],
    remainder="drop",
)

# Match the Part 1 selected model and tuning settings: Set 3 tree,
# max_depth=8 and min_samples_leaf=100. Preprocessing is fit on dev only.
part1_pipeline = Pipeline([
    ("prepare", PrepareTitleFeatures(genre_columns)),
    ("runtime", RuntimeMedianImputer()),
    ("encoding", encoding),
    ("model", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=100,
        random_state=SEED,
    )),
])

part1_pipeline.fit(X_dev, y_dev)
holdout_probability = part1_pipeline.predict_proba(X_holdout)[:, 1]
holdout_prediction = (holdout_probability >= THRESHOLD).astype(int)

def metric_row(name, truth, probability, prediction):
    precision = np.nan
    if prediction.sum() > 0:
        precision = 100 * precision_score(truth, prediction, zero_division=0)
    return {
        "Model": name,
        "Log loss": log_loss(truth, probability, labels=[0, 1]),
        "Misclassification (%)": 100 * (1 - accuracy_score(truth, prediction)),
        "Precision (%)": precision,
        "Recall (%)": 100 * recall_score(truth, prediction, zero_division=0),
    }

# Baseline predicts the development-set Netflix share for every holdout title.
baseline_probability = np.full(len(y_holdout), y_dev.mean())
baseline_prediction = (baseline_probability >= THRESHOLD).astype(int)

section2_results = pd.DataFrame([
    metric_row("Selected tree, Feature Set 3", y_holdout,
               holdout_probability, holdout_prediction),
    metric_row("Development-prevalence baseline", y_holdout,
               baseline_probability, baseline_prediction),
])

print("PROJECT 2, SECTION 2: HOLDOUT RESULTS")
print(section2_results.to_string(index=False))
print("\nPart 1 selected-tree CV log loss: 0.564583")
print("Part 1 baseline CV log loss: 0.662679")
section2_results.to_csv("part2_section2_holdout_metrics.csv", index=False)


PROJECT 2, SECTION 2: HOLDOUT RESULTS
                          Model  Log loss  Misclassification (%)  Precision (%)  Recall (%)
   Selected tree, Feature Set 3  0.560175              31.273094      59.854604    63.06383
Development-prevalence baseline  0.671448              39.469264            NaN     0.00000

Part 1 selected-tree CV log loss: 0.564583
Part 1 baseline CV log loss: 0.662679


In [8]:
# Project 2, Section 3(a)-(b): unpenalized logistic regression
# Inference specification: Set 3 predictors, age reference='all', movie reference
# (Type=0), with the same development-learned imputations applied to holdout.

age_levels = sorted(X_dev["Age"].fillna("unrated").astype(str).unique())
age_reference = "all" if "all" in age_levels else age_levels[0]
inference_numeric = ["runtime_minutes", "imdb_rating", "rt_score", "imdb_num_votes", "Year"]
inference_binary = ["Type", "runtime_missing", "genres_missing"] + genre_columns

def prepare_for_inference(fit_rows, rows_to_transform):
    cleaner = PrepareTitleFeatures(genre_columns)
    fit_clean = cleaner.transform(fit_rows)
    transform_clean = cleaner.transform(rows_to_transform)

    runtime_imputer = RuntimeMedianImputer().fit(fit_clean)
    fit_prepared = runtime_imputer.transform(fit_clean)
    transformed = runtime_imputer.transform(transform_clean)
    numeric_medians = {
        col: pd.to_numeric(fit_prepared[col], errors="coerce").median()
        for col in inference_numeric
    }
    return fit_prepared, transformed, numeric_medians


def inference_matrix(prepared, numeric_medians):
    x = pd.DataFrame(index=prepared.index)
    for col in inference_numeric:
        values = pd.to_numeric(prepared[col], errors="coerce")
        x[col] = values.fillna(numeric_medians[col])

    # Use readable units: runtime per 10 min, RT per 10 points, votes per 1,000.
    x["runtime_minutes"] /= 10
    x["rt_score"] /= 10
    x["imdb_num_votes"] /= 1000
    x["Year"] -= 2000

    for col in inference_binary:
        x[col] = pd.to_numeric(prepared[col], errors="coerce").fillna(0)

    age = prepared["Age"].fillna("unrated").astype(str)
    for level in age_levels:
        if level != age_reference:
            x[f"Age_{level}"] = (age == level).astype(int)
    return x.astype(float)


dev_prepared, holdout_prepared, dev_numeric_medians = prepare_for_inference(
    X_dev, X_holdout
)
X_dev_inference = inference_matrix(dev_prepared, dev_numeric_medians)
X_holdout_inference = inference_matrix(holdout_prepared, dev_numeric_medians)

def fit_logit(x, outcome):
    design = sm.add_constant(x, has_constant="add")
    return sm.Logit(outcome.astype(int), design).fit(
        method="lbfgs", maxiter=300, disp=False
    )


def coefficient_table(result):
    ci = result.conf_int()
    table = pd.DataFrame({
        "coefficient": result.params,
        "std_error": result.bse,
        "p_value": result.pvalues,
        "ci_95_low": ci[0],
        "ci_95_high": ci[1],
    })
    table["odds_ratio"] = np.exp(table["coefficient"])
    table["or_ci_95_low"] = np.exp(table["ci_95_low"])
    table["or_ci_95_high"] = np.exp(table["ci_95_high"])
    return table


# 3(a): development data.
development_logit = fit_logit(X_dev_inference, y_dev)
dev_coef = coefficient_table(development_logit)
print("PROJECT 2, SECTION 3(a): DEVELOPMENT RESULTS")
print(dev_coef.to_string())
dev_coef.to_csv("part2_section3a_development_logit.csv", index_label="term")

# 3(b): same feature specification and development-fitted imputation on holdout.
holdout_logit = fit_logit(X_holdout_inference, y_holdout)
holdout_coef = coefficient_table(holdout_logit)
comparison = dev_coef.add_prefix("development_").join(
    holdout_coef.add_prefix("holdout_"), how="outer"
)
print("\nPROJECT 2, SECTION 3(b): COEFFICIENT COMPARISON")
print(comparison.to_string())
holdout_coef.to_csv("part2_section3b_holdout_logit.csv", index_label="term")
comparison.to_csv("part2_section3b_coefficient_comparison.csv", index_label="term")

if not development_logit.mle_retvals.get("converged", True):
    print("WARNING: development logistic model did not converge.")
if not holdout_logit.mle_retvals.get("converged", True):
    print("WARNING: holdout logistic model did not converge.")


/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2620: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2679: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))


PROJECT 2, SECTION 3(a): DEVELOPMENT RESULTS
                   coefficient  std_error  p_value  ci_95_low  ci_95_high  odds_ratio  or_ci_95_low  or_ci_95_high
const                      0.0   0.175356      1.0  -0.343692    0.343692         1.0      0.709148       1.410144
runtime_minutes            0.0   0.008425      1.0  -0.016512    0.016512         1.0      0.983623       1.016650
imdb_rating                0.0   0.022762      1.0  -0.044613    0.044613         1.0      0.956368       1.045623
rt_score                   0.0   0.017138      1.0  -0.033590    0.033590         1.0      0.966968       1.034160
imdb_num_votes             0.0   0.000219      1.0  -0.000430    0.000430         1.0      0.999570       1.000430
Year                       0.0   0.001246      1.0  -0.002441    0.002441         1.0      0.997562       1.002444
Type                       0.0   0.067801      1.0  -0.132888    0.132888         1.0      0.875563       1.142122
runtime_missing            0.0   0.

/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2620: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2679: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:268: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  mlefit = super().fit(


In [ ]:
# Project 2, Section 3(c): nonparametric bootstrap CIs on development rows.
# Preprocessing medians are refit inside each bootstrap sample.
rng = np.random.default_rng(SEED)
bootstrap_parameters = []
bootstrap_failures = 0

for _ in range(BOOTSTRAP_REPS):
    sampled_indices = rng.integers(0, len(X_dev), size=len(X_dev))
    boot_x_raw = X_dev.iloc[sampled_indices].reset_index(drop=True)
    boot_y = y_dev.iloc[sampled_indices].reset_index(drop=True)

    try:
        boot_prepared, _, boot_medians = prepare_for_inference(boot_x_raw, boot_x_raw)
        boot_x = inference_matrix(boot_prepared, boot_medians)
        boot_fit = fit_logit(boot_x, boot_y)
        if boot_fit.mle_retvals.get("converged", True):
            bootstrap_parameters.append(boot_fit.params)
        else:
            bootstrap_failures += 1
    except Exception:
        bootstrap_failures += 1

if len(bootstrap_parameters) < 2:
    raise RuntimeError("Too few successful bootstrap fits; investigate model convergence.")

bootstrap_frame = pd.DataFrame(bootstrap_parameters)
bootstrap_ci = pd.DataFrame({
    "bootstrap_ci_95_low": bootstrap_frame.quantile(0.025),
    "bootstrap_ci_95_high": bootstrap_frame.quantile(0.975),
})
standard_ci = dev_coef[["ci_95_low", "ci_95_high"]].rename(columns={
    "ci_95_low": "standard_ci_95_low",
    "ci_95_high": "standard_ci_95_high",
})
ci_comparison = standard_ci.join(bootstrap_ci, how="outer")
print(f"Successful bootstrap fits: {len(bootstrap_parameters)}; failed: {bootstrap_failures}")
print(ci_comparison.to_string())
ci_comparison.to_csv("part2_section3c_bootstrap_vs_standard_ci.csv", index_label="term")


/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2620: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2679: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2620: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2679: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2620: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X))
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:2679: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(s

In [ ]:
# Project 2, Section 3(d): multiple testing adjustments.
# Intercept is excluded from the set of coefficient tests.
tested = dev_coef.drop(index="const", errors="ignore").copy()
tested["p_bh"] = multipletests(tested["p_value"], alpha=0.05, method="fdr_bh")[1]
tested["significant_bh_0.05"] = tested["p_bh"] < 0.05
tested["p_bonferroni"] = multipletests(tested["p_value"], alpha=0.05, method="bonferroni")[1]
tested["significant_bonferroni_0.05"] = tested["p_bonferroni"] < 0.05
print("PROJECT 2, SECTION 3(d): ADJUSTED P-VALUES")
print(tested[["p_value", "p_bh", "significant_bh_0.05",
              "p_bonferroni", "significant_bonferroni_0.05"]].to_string())
tested.to_csv("part2_section3d_multiple_testing.csv", index_label="term")
